# SegSure Exploratory Data Analysis

This notebook provides an exploratory analysis of segmentation disagreement between AtoMx and Proseg.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style('whitegrid')

## Load Results


In [ ]:
results_dir = Path('results')

# Load cell matches
matches = pd.read_csv(results_dir / 'tables' / 'cell_matches.csv')
print(f"Loaded {len(matches)} cell matches")
print(matches.head())

In [ ]:
# Load transcript metrics if available
tx_metrics_file = results_dir / 'tables' / 'transcript_metrics.csv'
if tx_metrics_file.exists():
    tx_metrics = pd.read_csv(tx_metrics_file)
    print(f"Loaded {len(tx_metrics)} transcript metrics")
    print(tx_metrics.head())
else:
    print("Transcript metrics not found")

In [ ]:
# Load neighborhood metrics if available
neigh_metrics_file = results_dir / 'tables' / 'neighborhood_metrics.csv'
if neigh_metrics_file.exists():
    neigh_metrics = pd.read_csv(neigh_metrics_file)
    print(f"Loaded {len(neigh_metrics)} neighborhood metrics")
    print(neigh_metrics.head())
else:
    print("Neighborhood metrics not found")

## Matching Summary


In [ ]:
# Basic statistics
print(f"Cell matching statistics:")
print(f"  Total matches: {len(matches)}")
print(f"  Mean distance: {matches['distance'].mean():.2f}")
print(f"  Median distance: {matches['distance'].median():.2f}")
print(f"  Max distance: {matches['distance'].max():.2f}")

In [ ]:
# Plot matching distance distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(matches['distance'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Centroid Distance')
ax.set_ylabel('Count')
ax.set_title('Distribution of Cell Matching Distances')
plt.tight_layout()
plt.show()

## Transcript Discord Analysis


In [ ]:
if 'tx_metrics' in locals() and 'jaccard_index' in tx_metrics.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Jaccard index distribution
    axes[0].hist(tx_metrics['jaccard_index'].dropna(), bins=30, 
                 color='green', alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Jaccard Index')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Transcript Assignment Overlap (Jaccard Index)')
    
    # Discord severity
    if 'discord_severity' in tx_metrics.columns:
        severity_counts = tx_metrics['discord_severity'].value_counts()
        severity_order = ['none', 'low', 'medium', 'high']
        severity_counts = severity_counts.reindex(
            [s for s in severity_order if s in severity_counts.index]
        )
        axes[1].bar(severity_counts.index, severity_counts.values, 
                   color=['green', 'yellow', 'orange', 'red'][:len(severity_counts)],
                   alpha=0.7, edgecolor='black')
        axes[1].set_xlabel('Discord Severity')
        axes[1].set_ylabel('Count')
        axes[1].set_title('Discord Severity Distribution')
    
    plt.tight_layout()
    plt.show()

## High Discord Regions


In [ ]:
if 'neigh_metrics' in locals() and 'neighbor_discord_density' in neigh_metrics.columns:
    high_discord = neigh_metrics[neigh_metrics['neighbor_discord_density'] > 0.5]
    print(f"High discord cells (density > 0.5): {len(high_discord)}")
    print(f"Percentage: {100 * len(high_discord) / len(neigh_metrics):.1f}%")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(neigh_metrics['neighbor_discord_density'].dropna(), bins=30,
           color='purple', alpha=0.7, edgecolor='black')
    ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='High discord threshold')
    ax.set_xlabel('Neighborhood Discord Density')
    ax.set_ylabel('Count')
    ax.set_title('Neighborhood Discord Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary Statistics


In [ ]:
print("=" * 60)
print("SEGSURE DISAGREEMENT ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nCell Matching:")
print(f"  Matched cell pairs: {len(matches)}")

if 'tx_metrics' in locals():
    print(f"\nTranscript Discord:")
    print(f"  Mean Jaccard Index: {tx_metrics['jaccard_index'].mean():.3f}")
    print(f"  Discord rates by severity:")
    if 'discord_severity' in tx_metrics.columns:
        for severity in ['high', 'medium', 'low', 'none']:
            count = (tx_metrics['discord_severity'] == severity).sum()
            if count > 0:
                print(f"    {severity}: {count} ({100*count/len(tx_metrics):.1f}%)")

if 'neigh_metrics' in locals():
    print(f"\nNeighborhood Discord:")
    high_count = (neigh_metrics['neighbor_discord_density'] > 0.5).sum()
    print(f"  High discord regions: {high_count}")
    print(f"  Mean neighbor discord density: {neigh_metrics['neighbor_discord_density'].mean():.3f}")

print("\n" + "=" * 60)